# YOLO26-P2 LLM-GE commands

Copy the Bash blocks for your current scenario in order. Mac commands run on the Mac; all other blocks run on PACE from the repository root. The desired model code and dependencies must already be in that checkout. Sections 1–8 launched the October 28, 2026 search. The October 30 screenshots show generation 2 saved, followed by a Slurm-status error; the 21-generation search was **not complete**. Sections 9–12 cover that interrupted run. Do not pull, sync dependencies, or regenerate Slurm scripts in the PACE checkout before resuming this tag, because its checkpoint fingerprint must match.

## 1. Connect from the Mac

```bash
ssh agupta3057@login-ice.pace.gatech.edu
```

## 2. Enter the repository on PACE

```bash
cd /storage/ice1/6/9/agupta3057/llm-guided-evolution-object-detection
module load uv
```

## 3. Set the YOLO26-P2 environment

```bash
export FRED_SEED_FAMILY=yolo26_p2 FRED_NUM_GENERATIONS=21
export LLMGE_PORT=8479 LLMGE_CLUSTER=pace-ice
export LOCAL=false LOCAL_LLM=true INFERENCE_SUBMISSION=false
export LLMGE_LLM_MODEL=mixtral SUBMIT_ISLAND_CONTROLLER=0
export HF_HUB_DISABLE_XET=1
unset FRED_EVENTS LLMGE_BOUNDED_SESSION LLMGE_MODEL_PATH
unset LLM_MAX_NEW_TOKENS LLMGE_EVAL_NO_PROGRESS_TIMEOUT_SECONDS
unset LLMGE_MIN_SERVER_REMAINING_SECONDS LLMGE_REAL_SQUEUE
```

## 4. Create a new tag

```bash
RUN_DATE=$(date +%Y%m%d)
RUN_SUFFIX=$(date +%H%M%S)
export FRED_RUN_TAG="yolo26_p2_${RUN_DATE}_eot0_trial${RUN_SUFFIX}"
echo "new tag: $FRED_RUN_TAG"
test ! -e "fred_runs/$FRED_RUN_TAG" && echo 'new tag OK'
```

## 5. Confirm the selected configuration

```bash
uv run python -c 'from src.cfg import constants as c; print(c.SEED_FAMILY, c.RUN_TAG, c.SEED_NETWORK, c.num_generations)'
```

Expected: yolo26_p2, the new tag, seed_yolo26n_p2.yaml, and 21.

## 6. Confirm search data and generate Slurm scripts

```bash
test -s sota/FRED_LLM_GE/phase1_detection/seeds/yolo11/data_cache/challenging/search_train_step50/data.yaml && echo 'training data ready'
test -s sota/FRED_LLM_GE/phase1_detection/seeds/yolo11/data_cache/challenging/search_val_step10/data.yaml && echo 'validation data ready'
mkdir -p run_job_outputs/{server,islands,evolution,evaluation}
uv run python slurm.py
```

Both data-ready messages and a successful slurm.py completion are required before submission.

## 7. Submit the LLM server

```bash
squeue -u "$USER" -o '%i %T %j %R %L'
SERVER_JOB_ID=$(sbatch --parsable --partition=coe-gpu --export=ALL server.sh)
export LLMGE_SERVER_JOB_ID=${SERVER_JOB_ID%%;*}
echo "server job: $LLMGE_SERVER_JOB_ID"
squeue -j "$LLMGE_SERVER_JOB_ID" -o '%i %T %R %L'
```

Wait for RUNNING, then follow the server log:

```bash
tail -F "run_job_outputs/server/slurm-$LLMGE_SERVER_JOB_ID.out"
```

Wait for Uvicorn to report that startup completed. Press Ctrl-C to leave tail; the server job keeps running.

## 8. Submit exactly one controller

```bash
RUN_JOB_ID=$(sbatch --parsable --export=ALL run.sh)
RUN_JOB_ID=${RUN_JOB_ID%%;*}
echo "controller job: $RUN_JOB_ID"
tail -F "run_job_outputs/islands/slurm-$RUN_JOB_ID.out"
```

Press Ctrl-C to leave tail without stopping the job. Record both job IDs and the run tag. Do not submit a second controller for the same tag.

## 9. Check the interrupted October 28 run (PACE)

The screenshots show generation 2 completed and the original controller job 6659620 stopped after a temporary Slurm standby error. In a new PACE shell, run:

```bash
cd /storage/ice1/6/9/agupta3057/llm-guided-evolution-object-detection
module load uv
export FRED_SEED_FAMILY=yolo26_p2
export FRED_RUN_TAG=yolo26_p2_20261028_eot0_trial182051
sacct -j 6659620 --format=JobID,State,ExitCode,Elapsed -P
squeue -u "$USER" -o '%i %T %j %R %L'
test -s "fred_runs/$FRED_RUN_TAG/checkpoints/checkpoint_gen_2.pkl" && test -s "fred_runs/$FRED_RUN_TAG/checkpoints/global_gen_2.pkl" && echo 'generation 2 checkpoint pair present'
cat "fred_runs/$FRED_RUN_TAG/status.json"
```

Continue only when the pair-present message appears and no controller for this tag is active. The status file may be stale after a traceback. Completion requires generation 20, status complete, and a successful controller exit.

## 10. Generate results and Pareto plots from saved generations (PACE)

Run after section 9 in the same shell. This reads the saved checkpoints; it does not resume the search or run held-out evaluation.

```bash
uv run python analysis/scripts/fred_pareto_graphs.py
ls -lh "analysis/results/analysis/fred_pareto/$FRED_RUN_TAG"
head -n 10 "analysis/results/analysis/fred_pareto/$FRED_RUN_TAG/pareto_models.csv"
```

The folder contains candidates.csv (all valid evaluated candidates), pareto_models.csv (three-objective front and model paths), pareto2d_gen_*.png, pareto2d_final.png, best_by_generation.png, and usually pareto2d_fronts.gif. With the currently confirmed checkpoint, pareto2d_final.png means **through generation 2**, not the final 21-generation front. These are search-validation results. Rerun this block after later generations finish to refresh the figures and tables.

## 11. Copy the plots, CSVs, and raw run data to the Mac

Run in a **Mac terminal** with the PACE VPN connected, after section 10 succeeds. This saves a folder beside the earlier YOLO26 trial results. The raw run copy may be large. Repeat after the search finishes to refresh it.

```bash
FRED_RUN_TAG=yolo26_p2_20261028_eot0_trial182051
PACE_ROOT=/storage/ice1/6/9/agupta3057/llm-guided-evolution-object-detection
LOCAL_ROOT="$HOME/Desktop/GP/llm-guided-evolution-object-detection/sota/FRED_LLM_GE/PACE-ICE Run Results/YOLO26-P2-LLM-GE-$FRED_RUN_TAG"
mkdir -p "$LOCAL_ROOT/pareto" "$LOCAL_ROOT/run"
rsync -av "agupta3057@login-ice.pace.gatech.edu:$PACE_ROOT/analysis/results/analysis/fred_pareto/$FRED_RUN_TAG/" "$LOCAL_ROOT/pareto/"
rsync -avP "agupta3057@login-ice.pace.gatech.edu:$PACE_ROOT/fred_runs/$FRED_RUN_TAG/" "$LOCAL_ROOT/run/"
```

The tagged run folder contains checkpoints, status, candidate files, and evaluation artifacts. Slurm logs are under the separate run_job_outputs folder on PACE.

## 12. Resume the same search from the saved checkpoint (PACE)

Use this in the **same PACE shell after section 9** only after its checkpoint pair is confirmed and no controller for this tag is active. The saved tag and family are already set. If you opened a new shell, run section 2 and the export lines in section 9 first. Skip sections 4–6: do not create a new tag, pull Git, run uv sync, or regenerate slurm.py.

```bash
export FRED_NUM_GENERATIONS=21
export LLMGE_PORT=8479 LLMGE_CLUSTER=pace-ice
export LOCAL=false LOCAL_LLM=true INFERENCE_SUBMISSION=false
export LLMGE_LLM_MODEL=mixtral SUBMIT_ISLAND_CONTROLLER=0
export HF_HUB_DISABLE_XET=1
unset FRED_EVENTS LLMGE_BOUNDED_SESSION LLMGE_MODEL_PATH
unset LLM_MAX_NEW_TOKENS LLMGE_EVAL_NO_PROGRESS_TIMEOUT_SECONDS
unset LLMGE_MIN_SERVER_REMAINING_SECONDS LLMGE_REAL_SQUEUE
squeue -u "$USER" -o '%i %T %j %R %L'
SERVER_JOB_ID=$(sbatch --parsable --partition=coe-gpu --export=ALL server.sh)
export LLMGE_SERVER_JOB_ID=${SERVER_JOB_ID%%;*}
echo "new server job: $LLMGE_SERVER_JOB_ID"
squeue -j "$LLMGE_SERVER_JOB_ID" -o '%i %T %R %L'
tail -F "run_job_outputs/server/slurm-$LLMGE_SERVER_JOB_ID.out"
```

Wait for Uvicorn startup to complete, then press Ctrl-C. If the server exits or reports an error, stop and inspect it. Then submit **one** controller:

```bash
RUN_JOB_ID=$(sbatch --parsable --export=ALL run.sh)
RUN_JOB_ID=${RUN_JOB_ID%%;*}
echo "new controller job: $RUN_JOB_ID"
tail -F "run_job_outputs/islands/slurm-$RUN_JOB_ID.out"
```

The controller should report loading generation 2 and start generation 3. Record both new job IDs. Ctrl-C only leaves tail. Repeat a fresh server/controller window if the search pauses before generation 20. To check this controller later from the PACE repository folder:

```bash
sacct -j "$RUN_JOB_ID" --format=JobID,State,ExitCode,Elapsed -P
tail -n 80 "run_job_outputs/islands/slurm-$RUN_JOB_ID.out"
cat "fred_runs/$FRED_RUN_TAG/status.json"
```

If this is a new shell, set the recorded RUN_JOB_ID and FRED_RUN_TAG first. Call the search complete only when the main controller job is COMPLETED with exit code 0:0 and status.json says status complete and last_completed_generation 20. Then rerun sections 10 and 11 for the complete plots and data.